# ELM corpus: elastic differential cross sections

Unlike KDUQ, CHUQ and Test, the ELM corpus is not defined by a table of EXFOR
subentries. It is a query -- every EXFOR data set for elastic nucleon scattering on a set of near-spherical targets between 10 and 200 MeV -- followed by a
sequence of human judgements: entries excluded as duplicates or as lacking
uncertainties, parses repaired by naming the right uncertainty columns, and individual
points corrected for apparent transcription errors.

Those judgements are the corpus. They are recorded in `nn_corpora.elm` with the reasons
given in the original ELM notebooks, and applied here.

Proton elastic data are stored as a ratio to Rutherford throughout, matching the other corpora in this repo. This is a departure from the original ELM notebooks, which keep absolute cross sections where EXFOR reports them that way; the conversion is exactly invertible given the tabulated energy and target.

In [ ]:
%matplotlib inline
from matplotlib import pyplot as plt

import numpy as np

from nn_corpora import elm, elm_curate, munge, plotting, serialize, spec

## Targets

The corpus is restricted to near-spherical nuclei, where a spherical optical model is
defensible. The cut is on the quadrupole deformation, and drops $^{42}$Ca and $^{44}$Ca.

In [ ]:
targets = spec.elm_targets()
print(f"{len(targets)} targets below beta2 = {spec.MAX_BETA2}:")
print(", ".join(f"{A if A else 'nat'}{__import__('periodictable').elements[Z].symbol}"
                for A, Z in sorted(targets, key=lambda t: (t[1], t[0]))))

## Query

Both `dXS/dA` and `dXS/dRuth` are requested for protons. `exfor_tools` removes the
absolute measurement wherever the same subentry also reports a ratio at the same energy,
keeping the ratio -- the form the experimenters normalised.

In [ ]:
nn = elm_curate.query_elastic(
    projectile=spec.PROJECTILES["neutron"], quantities=("dXS/dA",),
    targets=targets, einc_range=spec.ELM_ELASTIC_EINC_RANGE,
    min_num_pts=spec.ELM_MIN_NUM_PTS,
)
pp = elm_curate.query_elastic(
    projectile=spec.PROJECTILES["proton"], quantities=("dXS/dA", "dXS/dRuth"),
    targets=targets, einc_range=spec.ELM_ELASTIC_EINC_RANGE,
    min_num_pts=spec.ELM_MIN_NUM_PTS,
)
print(f"(n,n): {sum(len(m.data['dXS/dA'].entries) for m in nn.values())} entries parsed")
print(f"(p,p): {sum(len(m.data['dXS/dA'].entries) + len(m.data['dXS/dRuth'].entries) for m in pp.values())} entries parsed")

## Repair failed parses

An entry fails to parse when it reports several uncertainty columns and `exfor_tools`
cannot tell which is which. The recipes below come from reading each entry's EXFOR
`ERR-ANALYS` text, as recorded in the original notebooks.

In [ ]:
result = elm_curate.ElmSectorResult(sector="elastic_diff_xs")
elm_curate.repair_failed_parses(nn, ("dXS/dA",), result)
elm_curate.repair_failed_parses(pp, ("dXS/dA", "dXS/dRuth"), result)
print("\n".join(sorted(set(result.repaired))) or "nothing to repair")

## Exclusions

Entries rejected as duplicates, as lacking uncertainties, or as inconsistent with other
measurements of the same quantity. Every reason is the one recorded in the original
notebooks.

In [ ]:
elm_curate.exclude_entries(nn, "dXS/dA", elm.EXCLUDED_NN, result)
elm_curate.exclude_entries(pp, "dXS/dA", elm.EXCLUDED_PP_ABSOLUTE, result)
elm_curate.exclude_entries(pp, "dXS/dRuth", elm.EXCLUDED_PP_RUTHERFORD, result)
print("\n".join(sorted(set(result.excluded))))

## Point-level corrections

Found by plotting the data and looking: a mistranscribed point sits an order of
magnitude away from its neighbours. Each correction records what was changed.

In [ ]:
elm_curate.apply_point_fixes({**nn, **pp}, result)
elm_curate.apply_uncertainty_patches({**nn, **pp}, result)
elm_curate.apply_uncertainty_transplant(pp, result)
print("\n".join(sorted(set(result.repaired))))

## Munge and serialize

In [ ]:
elm_curate.finalize(nn, "elastic_diff_xs", "neutron", result)
elm_curate.finalize(pp, "elastic_diff_xs", "proton", result)
print(result.summary())

## Inspect

Plotted by target and grouped by incident energy, offset for legibility. This is the
view the corrections above were made from.

In [ ]:
for target, multi in sorted(pp.items())[:4]:
    measurements = [m for q in multi.data.values()
                    for e in q.entries.values() for m in e.measurements]
    if measurements:
        plotting.plot_angular(measurements, title=plotting._latex(
            f"{target[0] or 'nat'}{__import__('periodictable').elements[target[1]].symbol}"))
plt.show()

## Write

In [ ]:
serialize.write_sector(result.records, corpus="elm", sector="elastic_diff_xs",
                       bibtex=result.bibtex)
print(f"wrote {len(result.records)} measurements to data/elm/elastic_diff_xs/")
